## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks demonstrates how to access the TerraClimate dataset via the Microsoft Planetary Computer. . TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: [terraclimate- overview](https://planetarycomputer.microsoft.com/dataset/terraclimate#overview) 

## TerraClimate Data Extraction (Phase 2)

**Key Improvements for Phase 2:**
* Extraction of **Precipitation (pr)** and **Maximum Temperature (tmax)**.
* Optimized spatial join using **cKDTree** for faster processing.
* Checkpoint system to prevent data loss during long extractions.

## Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process.

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os
import time


## Extracting TerraClimate Data Using API Calls

The API-based method allows us to efficiently access **TerraClimate** data for specific regions and time periods through the [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/), ensuring scalability and reproducibility of the process.

Through the API, we can extract climate variables such as **Potential Evapotranspiration (PET)**, which represents the atmospheric demand for water. This variable provides important insights into surface moisture balance and helps improve the accuracy of water quality modeling.

This approach ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis.


### Loading and Mapping TerraClimate Data

This section demonstrates how **TerraClimate climate variables**, such as **Potential Evapotranspiration (PET)**, are loaded and mapped to sampling locations.

- The **load_terraclimate_dataset** function opens the TerraClimate Zarr/NetCDF dataset from the Microsoft Planetary Computer, handling storage options automatically.
- The **filterg** function filters the dataset for the desired time range (2011–2015) and the spatial extent corresponding to the study region. The resulting data is converted into a pandas DataFrame with standardized column names.
- The **assign_nearest_climate** function maps each sampling location to its **nearest TerraClimate grid point** using a KD-tree and assigns the climate variable values corresponding to the closest timestamp.

This workflow ensures efficient, reproducible retrieval of climate variables, while allowing participants to work with pre-extracted CSV files for faster benchmarking and analysis.


In [ ]:
# 1. FUNCTION TO CONNECT WITH MICROSOFT
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]
    
    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(asset.href, **asset.extra_fields["xarray:open_kwargs"])
    return ds

# 2. FUNCTION TO FILTER THE REGION (Multiple variables)
def filter_terraclimate_fase2(ds):
# Try to detect the names of the available variables
# Sometimes they are 'ppt' instead of 'pr', or 'pdsi', etc.

    available_vars = list(ds.data_vars)
    print(f"Variables detected on the server: {available_vars}")
    
# Security mapping: We look for the best available
# pr (precipitation) is usually 'ppt' or 'pr'
# pet (evapotranspiration) is usually 'pet'
# ro (runoff) is usually 'run' or 'ro'
    
    target_vars = []
    for v in ["pet", "pr", "ppt", "ro", "run", "tmax", "tmmx"]:
        if v in available_vars:
            target_vars.append(v)
    
    print(f"Extracting compatible variables: {target_vars}")
    
    ds_subset = ds[target_vars].sel(time=slice("2011-01-01", "2015-12-31"))

    df_list = []
    for i in tqdm(range(len(ds_subset.time)), desc="Filtrando meses"):
        df_month = ds_subset.isel(time=i).to_dataframe().reset_index()
        df_month_filter = df_month[
            (df_month['lat'] > -35.2) & (df_month['lat'] < -21.7) &
            (df_month['lon'] > 14.9) & (df_month['lon'] < 32.8)
        ].dropna(subset=target_vars, how='all')
        
        df_list.append(df_month_filter)

    df_final = pd.concat(df_list, ignore_index=True)
    df_final['month_key'] = df_final['time'].dt.strftime('%Y-%m')
    
    # We standardized names for the final model
    rename_dict = {"lat": "Latitude", "lon": "Longitude", "ppt": "pr", "run": "ro", "tmmx": "tmax"}
    return df_final.rename(columns=rename_dict)

# 3. FUNCTION TO ASSIGN CLIMATE (Ultra fast)
def assign_all_climate_fase2(sa_df, climate_df):
    sa_df = sa_df.copy()
    sa_df['month_key'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True).dt.strftime('%Y-%m')

    unique_coords = climate_df[['Latitude', 'Longitude']].drop_duplicates()
    tree = cKDTree(np.radians(unique_coords.values))
    dist, idx = tree.query(np.radians(sa_df[['Latitude', 'Longitude']].values), k=1)
    
    sa_df['nearest_lat'] = unique_coords.iloc[idx]['Latitude'].values
    sa_df['nearest_lon'] = unique_coords.iloc[idx]['Longitude'].values

    final_df = sa_df.merge(
        climate_df, 
        left_on=['nearest_lat', 'nearest_lon', 'month_key'],
        right_on=['Latitude', 'Longitude', 'month_key'],
        how='left'
    )
    return final_df[["pet", "pr", "ro", "tmax"]]


In [ ]:
import pystac_client
import planetary_computer as pc
import xarray as xr

def load_terraclimate_dataset():
    print("🔑 Generando nueva llave de acceso con Microsoft...")
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]
    
    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(asset.href, **asset.extra_fields["xarray:open_kwargs"])
    return ds

ds_global = load_terraclimate_dataset()
print("✅ Connection established.")

In [ ]:
import os
import time
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- SECURE PATH CONFIGURATION ---
# We use /tmp/ to avoid the "Read-only file system" error
checkpoint_path = "/tmp/terraclimate_temp_progress_fase2.csv"
archivo_entrenamiento = 'water_quality_training_dataset.csv'
archivo_template = 'submission_template.csv'

# 1. Load local files
Water_Quality_df = pd.read_csv(archivo_entrenamiento)
Validation_df = pd.read_csv(archivo_template)
if 'dis_detail_id' not in Water_Quality_df.columns:
    Water_Quality_df = Water_Quality_df.reset_index().rename(columns={'index': 'dis_detail_id'})

# 2. Rescue Logic (Checkpoint)
if os.path.exists(checkpoint_path):
    try:
        df_already_done = pd.read_csv(checkpoint_path)
        start_index = len(df_already_done['time'].unique())
        df_list = [df_already_done]
        print(f"🔄 RESCUE: Continuing from the month {start_index}...")
    except:
        print("⚠️ The rescue file was corrupted, starting from scratch.")
        df_list = []
        start_index = 0
else:
    df_list = []
    start_index = 0

# 3. Detect available variables
vars_fase2 = ["pet", "pr", "ppt", "ro", "run", "tmax", "tmmx"]
target_vars = [v for v in ds_global.data_vars if v in vars_fase2]

# 4. Extraction loop
print(f"🌦️ Extracting {target_vars}...")
try:
    # Define the time frame of the competition
    ds_subset = ds_global[target_vars].sel(time=slice("2011-01-01", "2015-12-31"))
    total_months = len(ds_subset.time)
    
    for i in tqdm(range(start_index, total_months), desc="Progreso"):
        # Extract one month
        df_month = ds_subset.isel(time=i).to_dataframe().reset_index()
        
        # Geographic filter South Africa
        df_month_filter = df_month[
            (df_month['lat'] > -35.2) & (df_month['lat'] < -21.7) &
            (df_month['lon'] > 14.9) & (df_month['lon'] < 32.8)
        ].dropna(subset=target_vars, how='all')
        
        df_list.append(df_month_filter)
        
        # Save every 5 months in the /tmp/ folder
        if i % 5 == 0:
            pd.concat(df_list, ignore_index=True).to_csv(checkpoint_path, index=False)
            
    # --- FINAL PROCESS ---
    print("🔗 Uniendo datos finales...")
    climate_full_df = pd.concat(df_list, ignore_index=True)
    climate_full_df['month_key'] = pd.to_datetime(climate_full_df['time']).dt.strftime('%Y-%m')
    rename_dict = {"lat": "Latitude", "lon": "Longitude", "ppt": "pr", "run": "ro", "tmmx": "tmax"}
    climate_full_df = climate_full_df.rename(columns=rename_dict)

    # Map to Training and Validation
    tc_train = assign_all_climate_fase2(Water_Quality_df, climate_full_df)
    tc_val = assign_all_climate_fase2(Validation_df, climate_full_df)

    # Save the final CSV files in the root folder
    pd.concat([Water_Quality_df[['dis_detail_id']], tc_train], axis=1).to_csv("terraclimate_features_training_fase2.csv", index=False)
    col_id_val = 'dis_detail_id' if 'dis_detail_id' in Validation_df.columns else 'ID'
    pd.concat([Validation_df[[col_id_val]], tc_val], axis=1).to_csv("terraclimate_features_validation_fase2.csv", index=False)
    
    print("✨ SUCCESSFULLY COMPLETED!✨")

except Exception as e:
    print(f"\n❌ ERROR: {e}")
    print("💡 Suggested action: If the error is related to Authentication, run Block 1 and restart this block.")

In [ ]:
from scipy.spatial import cKDTree
import pandas as pd
import numpy as np
import os

try:
    print("🔗 Consolidando datos descargados...")
    # 1. Combine the list of months in memory
    climate_full_df = pd.concat(df_list, ignore_index=True)
    
    # 2. Correct dates (Mutilating the time to avoid parsing errors)
    climate_full_df['time_clean'] = pd.to_datetime(climate_full_df['time'].astype(str).str[:10])
    climate_full_df['month_key'] = climate_full_df['time_clean'].dt.strftime('%Y-%m')

    # 3. Rename columns
    rename_dict = {"lat": "Latitude", "lon": "Longitude", "ppt": "pr", "run": "ro", "tmmx": "tmax"}
    climate_full_df = climate_full_df.rename(columns=rename_dict)
    
    # 4. Final variables
    vars_finales = [v for v in ["pet", "pr", "ro", "tmax"] if v in climate_full_df.columns]
    print(f"✅ Variables to be processed: {vars_finales}")

    # 5. Spatial Mapping
    def assign_resilient_final(sa_df, climate_df, vars_to_use):
        sa_copy = sa_df.copy()
        sa_copy['month_key'] = pd.to_datetime(sa_copy['Sample Date'], dayfirst=True).dt.strftime('%Y-%m')
        unique_coords = climate_df[['Latitude', 'Longitude']].drop_duplicates()
        tree = cKDTree(np.radians(unique_coords.values))
        dist, idx = tree.query(np.radians(sa_copy[['Latitude', 'Longitude']].values), k=1)
        sa_copy['nearest_lat'] = unique_coords.iloc[idx]['Latitude'].values
        sa_copy['nearest_lon'] = unique_coords.iloc[idx]['Longitude'].values
        return sa_copy.merge(
            climate_df[['Latitude', 'Longitude', 'month_key'] + vars_to_use], 
            left_on=['nearest_lat', 'nearest_lon', 'month_key'],
            right_on=['Latitude', 'Longitude', 'month_key'],
            how='left'
        )[vars_to_use]

    print("🛰️ Mapping data...")
    tc_train = assign_resilient_final(Water_Quality_df, climate_full_df, vars_finales)
    tc_val = assign_resilient_final(Validation_df, climate_full_df, vars_finales)

    # 6. SAVE IN /tmp/ (To avoid the Read-only error)
    print("💾 Saving files in /tmp/...")
    
    path_train = "/tmp/terraclimate_features_training_fase2.csv"
    path_val = "/tmp/terraclimate_features_validation_fase2.csv"
    
    pd.concat([Water_Quality_df[['dis_detail_id']], tc_train], axis=1).to_csv(path_train, index=False)
    
    col_id_val = 'dis_detail_id' if 'dis_detail_id' in Validation_df.columns else 'ID'
    pd.concat([Validation_df[[col_id_val]], tc_val], axis=1).to_csv(path_val, index=False)

    print(f"✨ SUCCESSFULLY COMPLETED! ✨")
    print(f"Your files are ready at:")
    print(f"1. {path_train}")
    print(f"2. {path_val}")
    
    # Verify that they exist
    if os.path.exists(path_train):
        print(f"\nConfirmed: The training file weighs {os.path.getsize(path_train) / 1024:.2f} KB")

except Exception as e:
    print(f"❌ Critical error: {e}")

In [ ]:
import os
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.spatial import cKDTree 

# --- TEMPORARY FOLDER PATH CONFIGURATION ---
# Moved EVERYTHING to /tmp/ to avoid the "Read-only file system"

checkpoint_path = "/tmp/terraclimate_temp_progress_fase2.csv"
output_train = "/tmp/terraclimate_features_training_fase2.csv"
output_val = "/tmp/terraclimate_features_validation_fase2.csv"

archivo_entrenamiento = 'water_quality_training_dataset.csv'
archivo_template = 'submission_template.csv'

# 1. Upload local files
Water_Quality_df = pd.read_csv(archivo_entrenamiento)
Validation_df = pd.read_csv(archivo_template)
if 'dis_detail_id' not in Water_Quality_df.columns:
    Water_Quality_df = Water_Quality_df.reset_index().rename(columns={'index': 'dis_detail_id'})

# 2. Rescue Logic (Checkpoint)
if os.path.exists(checkpoint_path):
    try:
        df_already_done = pd.read_csv(checkpoint_path)
        start_index = len(df_already_done['time'].unique())
        df_list = [df_already_done]
        print(f"🔄 RESCUE: Continuing from the month {start_index}...")
    except:
        df_list = []
        start_index = 0
else:
    df_list = []
    start_index = 0

# 3. Detect available variables
vars_fase2 = ["pet", "pr", "ppt", "ro", "run", "tmax", "tmmx"]
target_vars = [v for v in ds_global.data_vars if v in vars_fase2]

# 4. Extraction loop
print(f"🌦️ Extracting {target_vars}...")
try:
    ds_subset = ds_global[target_vars].sel(time=slice("2011-01-01", "2015-12-31"))
    total_months = len(ds_subset.time)
    
    for i in tqdm(range(start_index, total_months), desc="Progreso"):
        df_month = ds_subset.isel(time=i).to_dataframe().reset_index()
        
        # Geographic filter South Africa
        df_month_filter = df_month[
            (df_month['lat'] > -35.2) & (df_month['lat'] < -21.7) &
            (df_month['lon'] > 14.9) & (df_month['lon'] < 32.8)
        ].dropna(subset=target_vars, how='all')
        
        df_list.append(df_month_filter)
        
        if i % 5 == 0:
            pd.concat(df_list, ignore_index=True).to_csv(checkpoint_path, index=False)
            
    # --- FINAL PROCESS ---
    print("🔗 Uniendo datos finales...")
    climate_full_df = pd.concat(df_list, ignore_index=True)
    
    # Robust date correction (to avoid the "unconverted data" error)
    climate_full_df['month_key'] = pd.to_datetime(climate_full_df['time'].astype(str).str[:10]).dt.strftime('%Y-%m')
    
    rename_dict = {"lat": "Latitude", "lon": "Longitude", "ppt": "pr", "run": "ro", "tmmx": "tmax"}
    climate_full_df = climate_full_df.rename(columns=rename_dict)
    vars_finales = [v for v in ["pet", "pr", "ro", "tmax"] if v in climate_full_df.columns]

    # Integrated mapping function to avoid scope errors
    def assign_resilient(sa_df, climate_df, vars_to_use):
        sa_copy = sa_df.copy()
        sa_copy['month_key'] = pd.to_datetime(sa_copy['Sample Date'], dayfirst=True).dt.strftime('%Y-%m')
        u_coords = climate_df[['Latitude', 'Longitude']].drop_duplicates()
        tree = cKDTree(np.radians(u_coords.values))
        dist, idx = tree.query(np.radians(sa_copy[['Latitude', 'Longitude']].values), k=1)
        sa_copy['nearest_lat'] = u_coords.iloc[idx]['Latitude'].values
        sa_copy['nearest_lon'] = u_coords.iloc[idx]['Longitude'].values
        return sa_copy.merge(
            climate_df[['Latitude', 'Longitude', 'month_key'] + vars_to_use], 
            left_on=['nearest_lat', 'nearest_lon', 'month_key'],
            right_on=['Latitude', 'Longitude', 'month_key'], how='left')[vars_to_use]

    tc_train = assign_resilient(Water_Quality_df, climate_full_df, vars_finales)
    tc_val = assign_resilient(Validation_df, climate_full_df, vars_finales)

    # SAVE FINAL FILES IN /tmp/
    pd.concat([Water_Quality_df[['dis_detail_id']], tc_train], axis=1).to_csv(output_train, index=False)
    
    col_id_val = 'dis_detail_id' if 'dis_detail_id' in Validation_df.columns else 'ID'
    pd.concat([Validation_df[[col_id_val]], tc_val], axis=1).to_csv(output_val, index=False)
    
    print(f"✨ SUCCESSFULLY COMPLETED! ✨")
    print(f"Files saved in: {output_train} and {output_val}")

except Exception as e:
    print(f"\n❌ ERROR: {e}")

In [ ]:
try:
    print("🔗 Combining final data...")
    # 1. Consolidate the downloaded data
    climate_full_df = pd.concat(df_list, ignore_index=True)
    
    # 2. Clear dates
    climate_full_df['month_key'] = pd.to_datetime(climate_full_df['time'].astype(str).str[:10]).dt.strftime('%Y-%m')
    
    # 3. Rename climate columns
    rename_dict = {"lat": "Latitude", "lon": "Longitude", "ppt": "pr", "run": "ro", "tmmx": "tmax"}
    climate_full_df = climate_full_df.rename(columns=rename_dict)
    vars_finales = [v for v in ["pet", "pr", "ro", "tmax"] if v in climate_full_df.columns]

    # 4. Spatial Mapping (Ensuring functions exist)
    from scipy.spatial import cKDTree
    def assign_final_step(sa_df, climate_df, vars_to_use):
        sa_copy = sa_df.copy()
        sa_copy['month_key'] = pd.to_datetime(sa_copy['Sample Date'], dayfirst=True).dt.strftime('%Y-%m')
        u_coords = climate_df[['Latitude', 'Longitude']].drop_duplicates()
        tree = cKDTree(np.radians(u_coords.values))
        dist, idx = tree.query(np.radians(sa_copy[['Latitude', 'Longitude']].values), k=1)
        sa_copy['nearest_lat'] = u_coords.iloc[idx]['Latitude'].values
        sa_copy['nearest_lon'] = u_coords.iloc[idx]['Longitude'].values
        return sa_copy.merge(
            climate_df[['Latitude', 'Longitude', 'month_key'] + vars_to_use], 
            left_on=['nearest_lat', 'nearest_lon', 'month_key'],
            right_on=['Latitude', 'Longitude', 'month_key'], how='left')[vars_to_use]

    tc_train = assign_final_step(Water_Quality_df, climate_full_df, vars_finales)
    tc_val = assign_final_step(Validation_df, climate_full_df, vars_finales)

    # 5. Logic for ID Column 
    print("🔍 Searching for ID column in the files...")
    
    # For the Validation/Template file
    posibles_ids = ['dis_detail_id', 'ID', 'id', 'Id', 'index']
    col_id_val = next((c for c in posibles_ids if c in Validation_df.columns), Validation_df.columns[0])
    print(f"✅ Using '{col_id_val}' as the identifier column in validation.")

    # 6. SAVE FINAL FILES IN /tmp/
    output_train = "/tmp/terraclimate_features_training_fase2.csv"
    output_val = "/tmp/terraclimate_features_validation_fase2.csv"

    pd.concat([Water_Quality_df[['dis_detail_id']], tc_train], axis=1).to_csv(output_train, index=False)
    pd.concat([Validation_df[[col_id_val]], tc_val], axis=1).to_csv(output_val, index=False)
    
    print(f"✨ SUCCESSFULLY COMPLETED! ✨")
    print(f"Files saved in: {output_train} and {output_val}")

except Exception as e:
    print(f"❌ Final error: {e}")
    print(f"Available columns in the template: {Validation_df.columns.tolist()}")

In [ ]:
import base64
from IPython.display import HTML

def create_download_link(file_path, title="Download file"):
    with open(file_path, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    filename = os.path.basename(file_path)
    # Generate a stylish button so the link doesn't break.
    html = f'''
        <a download="{filename}" href="data:text/csv;base64,{b64}" target="_blank">
            <button style="padding:10px; background-color:#4CAF50; color:white; border:none; cursor:pointer; border-radius:5px; margin:5px;">
                {title}: {filename}
            </button>
        </a>
    '''
    return HTML(html)

# Create the buttons for both files
print("🚀 Click the buttons below to download:")
display(create_download_link('/tmp/terraclimate_features_training_fase2.csv', "1. Download Training"))
display(create_download_link('/tmp/terraclimate_features_validation_fase2.csv', "2. Download Validation"))